# 🎙️ استوديو الصوت المصري — Colab (GPU T4 مجاني)

**بنشغّل الموديل بنفسنا** (VoiceTut-TTS) بدل الـ Space — من غير كوتة ولا حدود، وبنستخدم خط أنابيب المشروع كامل (تطبيع مصري + mastering + MP3).

**الخطوات:**
1. من القائمة: `Runtime` → `Change runtime type` → **T4 GPU** → `Save`
2. شغّل الخلايا بالترتيب (Shift + Enter)
3. عدّل نصوص إعلاناتك في خلية الإعدادات
4. في الآخر هيتنزّلك `ads.zip` فيه كل الأصوات (WAV + MP3)

> الموديل: **VoiceTut-TTS** (مصري أصلي + code-switching، Apache-2.0 → تجاري مسموح).
> حجم الموديل صغير (≈3GB VRAM) فبيشتغل على T4 بسهولة.

In [ ]:
!nvidia-smi

In [ ]:
# تثبيت المكتبات (دقيقتين تقريبًا). Colab فيه torch جاهز.
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q voicetut-tts soundfile pyloudnorm scipy lameenc

In [ ]:
# (اختياري) اسحب المشروع كامل عشان نستخدم طبقة التطبيع + الـ mastering بتاعتنا.
# الريبو private — حطّ توكن GitHub لو معاك، أو سيب التوكار فاضي وعدّي الخلية.
import os

GITHUB_TOKEN = ""        # ✏️ اختياري
REPO = "pharmacygamila-ship-it/egyptian-voice-studio"

if GITHUB_TOKEN:
    !git clone -q https://{GITHUB_TOKEN}@github.com/{REPO}.git project
    %cd project
elif os.path.isdir("project"):
    %cd project
else:
    print("⏭️ مفيش كود المشروع — هنشتغل بالمطبّع الافتراضي بتاع الموديل.")

print("المسار الحالي:", os.getcwd())

In [ ]:
# تحميل الموديل
from voicetut_tts import VoiceTutTTS

MODEL_ID = "mohammedaly22/VoiceTut-TTS"
tts = VoiceTutTTS.from_pretrained(MODEL_ID)
print("✅ الموديل اتحمّل")

In [ ]:
# ⚙️ الإعدادات — عدّل هنا
SPEAKER = "Sayed"     # غيّره لأي صوت من القائمة تحت
STEPS = 64            # 8..64 (أعلى = أجود وأبطأ)
GUIDANCE = 2.5
SPEED = 0.95

SPEAKERS = ["Abdelrahman","Abdullah","Kamal","Hossam","Mohamed","Omar",
            "Sayed","Zaki","Aly","Essam","Ahmed",
            "Asmaa","Esraa","Hanan","Sarah","Yasmin","Omnia"]
print("الأصوات المتاحة:", ", ".join(SPEAKERS))

# ✏️ إعلاناتك
ADS = {
    "ad1": "جرب قهوة بون النهاردة! طعم أصلي، ريحة تجنن، وسعر مش هتلاقيه في أي مكان تاني.",
    "ad2": "خصم 50% على كل المنتجات النهاردة وبس!",
}

# ✏️ نطق مخصوص لبراندات/أسماء (اختياري)
EXTRA_LEXICON = {
    # "Ziad": "زياد",
    # "Nescafe": "نِسْكافِيه",
}

In [ ]:
# ✨ التطبيع المصري (أرقام/تواريخ/أوقات بالعامية) + تطبيق القاموس
import os, sys

# نضمن إن جذر المشروع على مسار الاستيراد لو اتسحب
if os.path.isdir("backend") and os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

try:
    from backend.text.normalize import normalize_for_tts

    def prepare(text):
        out = normalize_for_tts(text, diacritics=True)
        if EXTRA_LEXICON:
            tts.add_names(EXTRA_LEXICON)
        return out

    print("✅ بنستخدم خط أنابيب المشروع للتطبيع")
except Exception as e:  # noqa: BLE001
    print("⚠️ مفيش كود المشروع — بنستخدم تطبيع الموديل:", e)

    def prepare(text):
        return text

for k, v in ADS.items():
    print(k, "←", prepare(v))

In [ ]:
# 🎛️ توليد كل الإعلانات + mastering احترافي (-14 LUFS) + MP3 + zip
import os, zipfile
import numpy as np, soundfile as sf, pyloudnorm as pyln
from IPython.display import Audio, display

def master(path, target_lufs=-14.0, ceiling_db=-1.0, multiband="voice"):
    """سلسلة المشروع الكاملة لو متاحة، وإلا نسخة مبسّطة."""
    try:
        from backend.audio.postprocess import postprocess
        postprocess(path, path, target_lufs=target_lufs, true_peak_db=ceiling_db, multiband=multiband)
        return path
    except Exception:
        pass
    data, sr = sf.read(path, dtype="float64")
    try:
        loud = pyln.Meter(sr).integrated_loudness(data)
        if np.isfinite(loud):
            data = data * (10 ** ((target_lufs - loud) / 20))
    except Exception:
        pass
    peak = float(np.max(np.abs(data))) if data.size else 0.0
    ceil = 10 ** (ceiling_db / 20)
    if peak > ceil:
        data = data * (ceil / peak)
    sf.write(path, np.clip(data, -1, 1), sr)
    return path

def to_mp3(path, bitrate=192):
    try:
        from backend.audio.export import export
        return export(path, "mp3", bitrate=bitrate)
    except Exception:
        return path

os.makedirs("out", exist_ok=True)
paths = []
for name, text in ADS.items():
    out = f"out/{name}.wav"
    tts.synthesize(prepare(text), speaker=SPEAKER, output=out,
                   num_step=STEPS, guidance_scale=GUIDANCE, speed=SPEED, normalize=False)
    master(out, multiband="voice")
    paths.append(out)
    mp3 = to_mp3(out)
    if mp3 != out:
        paths.append(mp3)
    print("✓", out, "(-14 LUFS)")
    display(Audio(out))

with zipfile.ZipFile("ads.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, os.path.basename(p))
print("✅ اتجهّز ads.zip")

try:
    from google.colab import files
    files.download("ads.zip")
except Exception as e:
    print("نزّل ads.zip يدوياً من ملفات الجلسة:", e)

## 🧬 استنساخ صوت (بموافقة صاحب الصوت)
ارفع مقطع **10–30 ثانية** نضيف، واكتب النص الجديد.

In [ ]:
from google.colab import files
from IPython.display import Audio, display

up = files.upload()          # ارفع ملف wav
ref = list(up.keys())[0]

NEW_TEXT = "النص اللي عايز تنطقه بصوت ده"
tts.synthesize(
    prepare(NEW_TEXT),
    ref_audio=ref,
    ref_text="",             # لو كتبت نص المقطع بتحسّن الدقة
    output="out/clone.wav",
    num_step=STEPS, guidance_scale=GUIDANCE, speed=SPEED, normalize=False,
)
master("out/clone.wav")
display(Audio("out/clone.wav"))
files.download("out/clone.wav")

## 📝 ملاحظات
- لازم تشغّل على **GPU (T4)** وإلا هيكون بطيء جدًا.
- Colab مجاني للتوليد، بس **مش** مخصص إنك تسيبه سيرفر شغّال 24 ساعة (مخالف للشروط). للسيرفر الدائم استخدم `deploy/modal_app.py` أو Pod على RunPod/Vast.
- لو عايز **موديل خاص بيك** (فاين-تيون على صوت/دومين معيّن) شوف `finetune/README.md` — نفس مسار OmniVoice.
- ملاحظة مهمة: بنمرّر `normalize=False` للموديل لأننا بنطبّع النص بنفسنا (تطبيع الموديل مرتين بيشيل التشكيل).
- ⚖️ استخدم الأصوات بموافقة أصحابها، وممنوع الانتحال أو التضليل.